# Embeddings

Alle bisherigen Methoden hatten einen gemeinsamen blinden Fleck, den wir in Skript 04 benannt haben: In einer Bag-of-Words-Darstellung sind *Kanzlerin* und *Merkel* oder *Asylwerber* und *Flüchtling* so verschieden, wie zwei Wörter nur sein können. Ähnlichkeit setzt gemeinsame Zeichenketten voraus.

**Embeddings** ersetzen jedes Wort oder jeden Text durch einen dichten Vektor aus einigen hundert Zahlen. Diese Vektoren werden so gelernt, dass Einheiten, die in ähnlichen Kontexten vorkommen, ähnliche Vektoren erhalten. Das ist die *distributionelle Hypothese*: Ein Wort wird durch die Gesellschaft charakterisiert, in der es auftritt. Embedding-Modelle sind neuronale Netze, aber sie sind **Encoder**, keine Generatoren - sie bilden Text auf Vektoren ab und erzeugen selbst keinen Text. Große Sprachmodelle (LLMs) folgen in Skript 07.

Am Ende dieses Skripts kannst du:

- mit **Wortvektoren** verwandte Wörter finden und erkennen, wo sie danebenliegen;
- **Dokumentvektoren** bilden, zuerst durch Mittelwertbildung, dann mit einem **Sentence Transformer**;
- eine **semantische Suche** durchführen, die Treffer ohne gemeinsame Wörter findet;
- die **Zuordnung von Meldungen** (Story Matching) aus Skript 04 und den **Klassifikator** aus Skript 05 mit Embeddings wiederholen und messen, was sich verändert hat.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket für linguistische Annotation und Wortvektoren
import spacy

# Pakete für Merkmalsextraktion, Klassifikation und Evaluierung
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

# Pakete für Sentence Transformer und die Projektion in zwei Dimensionen
import torch
import umap
from sentence_transformers import SentenceTransformer

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

Die Warnung von `tqdm` (einem Paket für Fortschrittsbalken) ist harmlos: Ohne das Paket `ipywidgets` zeigt Jupyter die Balken als Text statt als Grafik an. Das Ergebnis ist davon nicht betroffen.

## Daten

Wir arbeiten wieder mit den Schlagzeilen von Der Standard und Süddeutscher Zeitung, die wir mit `tm.load_corpus()` laden.

In [ ]:
# Lade das Korpus
corpus = tm.load_corpus()

## Wortvektoren

Die deutschen Modelle von spaCy enthalten statische Wortvektoren: einen 300-dimensionalen Vektor pro Wort, gelernt aus einer großen Sammlung deutscher Texte. "Statisch" bedeutet, dass ein Wort überall, wo es vorkommt, denselben Vektor hat.

Wir beginnen mit einer Falle. Das mittlere Modell aus Skript 02 *hat* Vektoren, aber sieh dir an, wie viele:

In [ ]:
# Lade das mittlere Modell und betrachte seine Vektoren
nlp_md = spacy.load("de_core_news_md")
v = nlp_md.vocab.vectors
print(f"de_core_news_md: {v.n_keys:,} Wörter teilen sich {v.shape[0]:,} verschiedene Vektoren")

# Suche die nächsten Nachbarn von "Kanzler"
keys, _, scores = v.most_similar(np.asarray([nlp_md.vocab["Kanzler"].vector]), n=4)
print("nächste Nachbarn von 'Kanzler':", [(nlp_md.vocab.strings[k], round(float(s), 2)) for k, s in zip(keys[0], scores[0])])

Um klein zu bleiben, bildet das mittlere Modell eine halbe Million Wörter auf zwanzigtausend Vektoren ab. Nicht verwandte Wörter *teilen* sich also einen Vektor und erscheinen als perfekte Nachbarn mit Ähnlichkeit 1.00. Für das, wofür `md` die Vektoren intern verwendet, ist das in Ordnung - für die Untersuchung von Wortbedeutungen ist es nutzlos. Prüfe immer, was deine Vektoren sind, bevor du Schlüsse daraus ziehst.

Das große Modell, `de_core_news_lg`, hat für jedes seiner 500.000 Wörter einen eigenen Vektor. Installiere es einmalig mit `python -m spacy download de_core_news_lg`. Wir benötigen hier nur seine Vektoren, daher lassen wir die übrigen Komponenten der Pipeline weg.

In [ ]:
# Lade das große Modell, nur mit den Komponenten, die wir für die Vektoren brauchen
nlp = spacy.load("de_core_news_lg", exclude=["parser", "ner", "lemmatizer", "attribute_ruler"])
vectors = nlp.vocab.vectors
print(f"de_core_news_lg: {vectors.n_keys:,} Wörter, {vectors.shape[0]:,} verschiedene Vektoren mit {vectors.shape[1]} Dimensionen")


# Finde die n nächsten Nachbarn eines Worts (ohne Varianten in anderer Groß-/Kleinschreibung)
def neighbours(word: str, n: int = 10) -> list[str]:
    keys, _, scores = vectors.most_similar(np.asarray([nlp.vocab[word].vector]), n=n * 3)
    seen, out = {word.casefold()}, []
    for key, score in zip(keys[0], scores[0]):
        w = nlp.vocab.strings[key]
        if w.casefold() not in seen:
            seen.add(w.casefold())
            out.append(f"{w} ({score:.2f})")
    return out[:n]


# Zeige die Nachbarn für einige Beispielwörter
pd.DataFrame({w: neighbours(w) for w in ["Kanzlerin", "Flüchtling", "Börse", "Wien", "Angriff"]})

Die Nachbarn sind verwandte Wörter, die ein Bag-of-Words-Modell nie miteinander verbinden könnte. Drei Dinge fallen auf:

- **Die Schreibweise schlägt durch.** Viele Nachbarn sind Flexionsformen und Komposita des Suchworts (*Flüchtlings*, *Kriegsflüchtling*, *Gegenangriff*, *Aktienbörse*). Diese Vektoren wurden mit Information über Wortteile trainiert. Das hilft bei der deutschen Morphologie, bedeutet aber auch, dass Wörter mit gemeinsamem Stamm einander ähneln, egal was sie bedeuten.
- **Verwandt ist nicht dasselbe wie ähnlich.** Die Nachbarn von *Wien* sind andere österreichische Städte und Wiener Bezirke, keine Synonyme für Wien; unter den Nachbarn von *Kanzlerin* stehen *Merkel*, aber auch Politikerinnen anderer Parteien. Wörter, die in denselben *Kontexten* vorkommen, liegen nahe beieinander, und Alternativen teilen sich Kontexte. Für die Suche kann das helfen oder schaden.
- **Die Vektoren sind in der Zeit eingefroren.** Sie wurden aus einer allgemeinen Sammlung deutscher Webtexte gelernt, nicht aus unserem Korpus. Was ein Wort in den Nachrichten eines bestimmten Jahres bedeutete - wofür etwa *Obergrenze* oder *Balkanroute* 2015 standen -, wissen sie nur, wenn es in ihren Trainingstexten schon so vorkam.

Und noch einmal das Problem der Komposita:

In [ ]:
# Prüfe, ob es für diese Wörter einen Vektor gibt
for word in ["Flüchtling", "Flüchtlingskrise", "Flüchtlingsunterkunft", "Flüchtlingsobergrenze", "Willkommenskultur", "Sachsen-Anhalt"]:
    print(f"{word:18} hat Vektor: {nlp.vocab[word].has_vector}")

Selbst eine halbe Million Wörter ist ein festes Vokabular. Ein neues Kompositum, das in den Nachrichten eines Jahres geprägt wurde, hat überhaupt keinen Vektor und trägt nichts bei - und im Deutschen werden ständig neue Komposita gebildet.

## Dokumentvektoren durch Mittelwertbildung

Der einfachste Dokumentvektor ist der Mittelwert seiner Wortvektoren. spaCy berechnet ihn als `doc.vector`.

In [ ]:
# Drei Umschreibungen derselben Meldung und eine Börsenmeldung
headlines = [
    "Kanzlerin sagt den Ländern mehr Geld für Flüchtlinge zu",
    "Merkel will Bundesländer bei Asylkosten entlasten",
    "Regierungschefin verspricht zusätzliche Mittel zur Unterbringung von Asylbewerbern",
    "Dax schließt mit deutlichem Minus",
]
docs = [nlp(h) for h in headlines]

# Vergleiche die Ähnlichkeit zur ersten Schlagzeile: TF-IDF gegen gemittelte Wortvektoren
tfidf_demo = TfidfVectorizer().fit(headlines + corpus["title"].tolist())
pd.DataFrame(
    {
        "TF-IDF": cosine_similarity(tfidf_demo.transform(headlines))[0].round(2),
        "gemittelte Wortvektoren": [docs[0].similarity(d) for d in docs],
    },
    index=headlines,
).round(2)

Die ersten drei Schlagzeilen sagen fast dasselbe mit anderen Worten. TF-IDF sieht überhaupt keine Überschneidung. Gemittelte Wortvektoren erkennen den Zusammenhang und setzen die Börsenmeldung deutlich darunter - aber die ausführlichste Umschreibung, die dritte, liegt nur knapp über der Mitte. Die Mittelwertbildung verwischt jeden Text in Richtung einer allgemeinen "Nachrichten"-Richtung, ignoriert die Wortreihenfolge und kann nicht gewichten, welche Wörter wichtig sind.

## Sentence Transformer

Ein **Sentence Transformer** ist ein neuronaler Encoder, der gezielt darauf trainiert wurde, Texte mit gleicher Bedeutung auf nahe beieinanderliegende Vektoren abzubilden. Er liest den ganzen Satz im Kontext, sodass der Beitrag eines Worts von seinen Nachbarn abhängt, und er kommt mit unbekannten Komposita zurecht, indem er sie in bekannte Wortteile zerlegt.

In diesem Kurs verwenden wir `paraphrase-multilingual-MiniLM-L12-v2`: klein, schnell und auf Paraphrasen in über fünfzig Sprachen trainiert, darunter Deutsch. Beim ersten Ausführen wird das Modell (etwa 0,5 GB) vom Hugging Face Hub heruntergeladen, danach wird es aus dem lokalen Cache geladen. Eine Grafikkarte (GPU) hilft, ist aber für ein Korpus dieser Größe nicht nötig.

In [ ]:
# Verwende die Grafikkarte, falls vorhanden, sonst die CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("läuft auf", device)

# Kodiere die Beispiel-Schlagzeilen und vergleiche sie mit der ersten
E_demo = encoder.encode(headlines, normalize_embeddings=True)
pd.Series((E_demo @ E_demo[0]).round(2), index=headlines, name="Sentence Transformer").to_frame()

Die drei Paraphrasen liegen jetzt alle klar über der nicht verwandten Börsenmeldung, deren Ähnlichkeit auf null fällt, und die ausführlichste Umschreibung rückt nach vorne. Jede Schlagzeile ist zu einem 384-dimensionalen Einheitsvektor geworden, und die Ähnlichkeit ist wieder ein Skalarprodukt. Das gesamte Korpus zu kodieren dauert auf einer CPU wenige Minuten, mit einer GPU Sekunden:

In [ ]:
# Kodiere alle Schlagzeilen des Korpus
E = encoder.encode(corpus["title"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
print(E.shape)

Das Ergebnis ist eine Matrix mit einer Zeile je Schlagzeile und 384 Spalten, den Dimensionen der Embeddings.

## Semantische Suche

Sind alle Schlagzeilen eingebettet, wird die Suche zu einer Nächste-Nachbarn-Suche: Wir betten die Suchanfrage ein und ordnen die Dokumente nach ihrer Ähnlichkeit. Zum Vergleich führen wir dieselbe Anfrage auch mit TF-IDF durch:

In [ ]:
# Erstelle die TF-IDF-Matrix der Schlagzeilen
tfidf = TfidfVectorizer(min_df=1, token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")
X = tfidf.fit_transform(corpus["title"])


# Zeige die n besten Treffer der Embedding-Suche und der TF-IDF-Suche nebeneinander
def search(query: str, n: int = 6) -> pd.DataFrame:
    emb = E @ encoder.encode([query], normalize_embeddings=True)[0]
    lex = cosine_similarity(tfidf.transform([query]), X).ravel()
    top_e, top_l = emb.argsort()[::-1][:n], lex.argsort()[::-1][:n]
    return pd.DataFrame({
        "Embedding": [f"{emb[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_e],
        "TF-IDF": [f"{lex[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_l],
    })


search("Ein Autobauer hat bei den Abgaswerten seiner Fahrzeuge betrogen")

Die Anfrage beschreibt den VW-Abgasskandal vom Herbst 2015, ohne die Schlagwörter zu verwenden, mit denen die Zeitungen darüber geschrieben haben (*VW*, *Volkswagen*, *Diesel*, *Abgasaffäre*). TF-IDF findet Treffer nur dort, wo ein Wort der Anfrage wörtlich vorkommt: einige Schlagzeilen mit *Abgaswerten*, aber auch den FC Bayern, der sich *betrogen* fühlt, und Google, das kein *Autobauer* werden will. Die Embedding-Suche findet durchgehend Schlagzeilen über manipulierte Abgaswerte, also das eigentliche Thema, auch wenn sie kein Wort mit der Anfrage teilen. Das ist der zentrale Vorteil von Embeddings: Sie vergleichen **Bedeutung statt Schreibweise**.

Da das Modell mehrsprachig ist, muss die Anfrage nicht einmal auf Deutsch sein:

In [ ]:
# Eine englische Anfrage an die deutschen Schlagzeilen
search("Refugees are walking along the Balkan route towards Germany")

Die englische Anfrage liefert deutsche Schlagzeilen - etwas, das keine wortbasierte Methode kann; TF-IDF findet höchstens Schlagzeilen, die zufällig ein englisches Wort enthalten. Dasselbe funktioniert mit anderen Sprachen, die das Modell kennt:

In [ ]:
# Eine französische Anfrage zum britischen EU-Referendum
search("Le Royaume-Uni vote pour quitter l'Union européenne")

Die französische Anfrage zum britischen EU-Referendum findet die Brexit-Schlagzeilen beider Zeitungen. Ein gemeinsamer Vektorraum für viele Sprachen bedeutet, dass sich Korpora in verschiedenen Sprachen vergleichen und durchsuchen lassen, ohne sie vorher zu übersetzen - etwa Der Standard und die SZ mit einer englischen oder französischen Zeitung desselben Jahres.

## Story Matching, noch einmal

In Skript 04 haben wir Schlagzeilen von Der Standard mit TF-IDF Schlagzeilen der Süddeutschen Zeitung aus demselben Monat zugeordnet und festgestellt, dass die meisten Standard-Schlagzeilen kein nahes Gegenstück hatten. Wir wiederholen genau dasselbe Verfahren mit Embeddings und lesen dieselben Ähnlichkeitsbereiche:

In [ ]:
# Positionen der Standard- und der SZ-Schlagzeilen und der Monat jeder Schlagzeile
std = np.flatnonzero(corpus["outlet"] == "Der Standard")
sz = np.flatnonzero(corpus["outlet"] == "Süddeutsche Zeitung")
month = corpus["month"].to_numpy()

# Für jeden Monat: Ähnlichkeiten der Embeddings und bester SZ-Treffer je Standard-Schlagzeile
rows = []
for m in np.unique(month):
    a, b = std[month[std] == m], sz[month[sz] == m]
    S = E[a] @ E[b].T
    best = S.argmax(axis=1)
    rows.append(pd.DataFrame({"std": a, "sz": b[best], "similarity": S[np.arange(len(a)), best]}))
pairs = pd.concat(rows, ignore_index=True)
pairs["standard"] = corpus["title"].to_numpy()[pairs["std"]]
pairs["sueddeutsche"] = corpus["title"].to_numpy()[pairs["sz"]]

# Zeige je Ähnlichkeitsbereich einige zufällige Paare
for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- Ähnlichkeit {lo:.1f}-{hi:.1f}: {len(band)} Paare")
    for _, r in band.sample(min(4, len(band)), random_state=0).iterrows():
        print(f"  STD: {r['standard']}\n  SZ:  {r['sueddeutsche']}\n")

Auch hier können wir die Ressorts als zweite Meinung heranziehen, wie in Skript 04: Wie oft stehen die beiden Schlagzeilen eines Paars in derselben Ressortgruppe?

In [ ]:
# Anteil der Paare aus derselben Ressortgruppe je Ähnlichkeitsband (nur Paare mit Gruppe auf beiden Seiten)
group = corpus["section_group"].to_numpy()
both = pairs.assign(group_std=group[pairs["std"]], group_sz=group[pairs["sz"]]).dropna(subset=["group_std", "group_sz"])
bands = pd.cut(both["similarity"], [0, 0.5, 0.6, 0.7, 0.8, 1.01], right=False)
agreement = (both["group_std"] == both["group_sz"]).groupby(bands, observed=True).agg(["mean", "size"])
agreement.columns = ["gleiche Ressortgruppe", "Paare"]
agreement.style.format({"gleiche Ressortgruppe": "{:.0%}"})

Zwei Unterschiede zu Skript 04 fallen auf. Erstens ist die **Skala** eine andere: Die Ähnlichkeiten der Embeddings sind durchwegs höher, weil zwei beliebige Schlagzeilen viel Bedeutung teilen ("jemandem ist irgendwo etwas passiert"); die allermeisten Standard-Schlagzeilen finden im selben Monat einen SZ-Partner über 0,5. Ein Schwellenwert, der für TF-IDF gewählt wurde, lässt sich daher nicht übertragen. Für jede neue Repräsentation muss der Schwellenwert durch Lesen neu kalibriert werden - und hier zeigt das Lesen, dass selbst im obersten Band viele Paare nicht dieselbe Geschichte erzählen. Die Übereinstimmung der Ressortgruppen steigt mit der Ähnlichkeit, ähnlich wie bei TF-IDF.

Zweitens sind die **Fehler** andere. Falsche Zuordnungen bei TF-IDF hatten ein Wort gemeinsam; falsche Zuordnungen bei Embeddings haben eine *Art von Meldung* gemeinsam - zwei verschiedene Unwetter, zwei verschiedene Nachrichten aus Syrien. Die Embeddings finden Paraphrasen, die TF-IDF übersehen hat (*Clooney bei Merkel* und *George Clooney trifft Angela Merkel*), und machen Fehler, die TF-IDF nie machen würde. Keines der beiden Verfahren ist einfach besser; ein robustes Zuordnungsverfahren kombiniert oft beide Signale und ein engeres Zeitfenster, als es die Monatsangaben der SZ erlauben.

## Eine Landkarte des Korpus

**UMAP** projiziert hochdimensionale Vektoren in zwei Dimensionen und versucht dabei, Nachbarn beisammen zu halten. Es ist ein Werkzeug zur Visualisierung: Abstände zwischen weit entfernten Clustern in der Grafik sind nicht aussagekräftig, nur die lokalen Nachbarschaften sind es.

In [ ]:
# Ein fester random_state macht das Layout reproduzierbar, dafür läuft UMAP nur auf
# einem Kern; n_jobs=1 sagt das ausdrücklich, statt eine Warnung auszulösen.
coords = umap.UMAP(n_neighbors=20, min_dist=0.1, metric="cosine", random_state=0, n_jobs=1).fit_transform(E)

# Zeichne die projizierten Schlagzeilen zweimal: eingefärbt nach Zeitung und nach Ressortgruppe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
for outlet in ["Süddeutsche Zeitung", "Der Standard"]:  # die kleinere Zeitung zuletzt, damit sie sichtbar bleibt
    idx = corpus.index[corpus["outlet"] == outlet]
    ax1.scatter(coords[idx, 0], coords[idx, 1], s=2, alpha=0.3, color=tm.OUTLET_COLOURS[outlet], label=outlet)
group_colours = {"Politik": "#2F6690", "Wirtschaft": "#2E7D5B", "Sport": "#B33A3A", "Panorama": "#946200",
                 "Digital": "#6A4C93"}
rest = corpus["section_group"].isna()
ax2.scatter(coords[rest, 0], coords[rest, 1], s=2, alpha=0.2, color="0.8", label="ohne Gruppe")
for name, colour in group_colours.items():
    idx = corpus.index[corpus["section_group"] == name]
    ax2.scatter(coords[idx, 0], coords[idx, 1], s=2, alpha=0.3, color=colour, label=name)
ax1.set(xticks=[], yticks=[], title="eingefärbt nach Zeitung")
ax2.set(xticks=[], yticks=[], title="eingefärbt nach Ressortgruppe")
for ax in (ax1, ax2):
    ax.legend(markerscale=6, frameon=False, fontsize=8)
fig.suptitle("Embeddings der Schlagzeilen, mit UMAP projiziert", fontweight="bold")
fig.tight_layout()
plt.show()

Vergleiche das mit der LSA-Grafik in Skript 04, in der die Dokumente auf wenige lange Arme verteilt waren. Hier zerfällt das Korpus in viele Nachbarschaften, und die rechte Grafik zeigt, dass sie zu einem guten Teil den Ressorts entsprechen, obwohl der Encoder die Ressorts nie gesehen hat: Sport bildet eine große eigene Region, Digital eine kleinere, Politik verteilt sich auf viele Inseln einzelner Themen und Geschichten. Links sieht man, dass sich die beiden Zeitungen in diesen Nachbarschaften durchmischen: Schlagzeilen über dieselbe Art von Ereignis landen beieinander, egal aus welcher Redaktion sie stammen. Wo die rote Farbe dichter wird, liegen Themen, die Der Standard stärker abdeckt, etwa die Technik seines großen Web-Ressorts.

## Embeddings als Merkmale für einen Klassifikator

Zum Schluss die Aufgabe aus Skript 05. Statt TF-IDF-Merkmalen geben wir das Embedding jeder Schlagzeile in eine logistische Regression, mit derselben zeitlichen Aufteilung in Trainings- und Testdaten. Die logistische Regression kennen wir bereits aus dem Skript `03-data-mining-classification`.

In [ ]:
# Lade Trainings- und Testdaten; ihre Embeddings haben wir oben schon berechnet (Zuordnung über doc_key)
train, test = tm.subject_task()
position = pd.Series(np.arange(len(corpus)), index=corpus["doc_key"])
E_train, E_test = E[position[train["doc_key"]].to_numpy()], E[position[test["doc_key"]].to_numpy()]

# Trainiere die logistische Regression auf den Embeddings
clf = LogisticRegression(max_iter=3000, C=4).fit(E_train, train["label"])
pred_emb = clf.predict(E_test)

# Die beste Pipeline aus Skript 05, hier neu trainiert, damit der Vergleich in sich geschlossen ist.

lesson05 = make_pipeline(
    FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                  token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LinearSVC(C=0.25),
).fit(train["title"], train["label"])
pred_tfidf = lesson05.predict(test["title"])

# Vergleiche Accuracy und Makro-F1 der beiden Modelle
pd.DataFrame(
    {
        "Skript 05: TF-IDF + SVM": [accuracy_score(test["label"], pred_tfidf), f1_score(test["label"], pred_tfidf, average="macro")],
        "Embeddings + logistische Regression": [accuracy_score(test["label"], pred_emb), f1_score(test["label"], pred_emb, average="macro")],
    },
    index=["Accuracy", "Makro-F1"],
).T.round(3)

Die beiden Modelle liegen praktisch gleichauf - bemerkenswert, wenn man bedenkt, dass das Embedding-Modell eine einfache logistische Regression ohne Tuning ist, die gegen eine abgestimmte Pipeline antritt, und dass der Encoder nie darauf trainiert wurde, Ressorts zu unterscheiden. Zwei Beobachtungen gelten allgemeiner:

- **Embeddings bringen Vorwissen mit, dünnbesetzte Merkmale lernen auswendig.** Ein Embedding-Modell bringt Wissen aus seinem Vortraining mit und kommt daher mit Wörtern zurecht, die in den gelabelten Daten fehlen. Die Testdaten stammen aus einem späteren Zeitraum mit neuen Meldungen und neuen Namen - genau dort zählt das. TF-IDF-Merkmale kennen nur die Schlagzeilen aus dem Training: Sie erfassen die Namen und Formate, die *damals* die Labels vorhergesagt haben.
- **Sie scheitern auf unterschiedliche Weise** und ergänzen sich daher. Ein Blick auf die Fälle, in denen die beiden Modelle uneins sind, lohnt sich:

In [ ]:
# Wähle die Schlagzeilen aus, bei denen die beiden Modelle unterschiedlich entscheiden
disagree = test.assign(tfidf=pred_tfidf, embedding=pred_emb)
disagree = disagree[disagree["tfidf"] != disagree["embedding"]]
print(f"die beiden Modelle sind sich bei {len(disagree)} von {len(test)} Schlagzeilen der Testdaten uneins")
disagree[["title", "label", "tfidf", "embedding"]].sample(10, random_state=0)

Die Modelle sind sich bei rund einem Drittel der Testdaten uneins, obwohl sie fast gleich oft richtig liegen. Viele Unstimmigkeiten gehen auf die Unschärfe der Ressorts aus Skript 05 zurück - die Buwog-Affäre um einen früheren Finanzminister zwischen *Wirtschaft* und *Inland*, die halbleeren Flüchtlingsunterkünfte zwischen *Panorama*, *Sonstiges* und *Wirtschaft*, eine Sonderkommission des Justizministers nach einem Mord zwischen *Panorama*, *Sonstiges* und *Inland* -, wobei die beiden Modelle auf verschiedenen Seiten landen. Ein Modell, das nur die Schlagzeile sieht, kann weiterhin nicht wissen, welches Ressort einen Artikel betreut hat, und dieser kleine Encoder wurde für Paraphrasen trainiert, nicht für die Klassifikation von Nachrichten. Dass sie verschieden scheitern, ist eine Chance: Übung 1 kombiniert die beiden Signale. Größere Encoder oder das Feintuning eines Encoders auf den gelabelten Schlagzeilen würden weiter helfen - zu echten Kosten an Rechenaufwand und Komplexität.

## Zusammenfassung

| Repräsentation | Erfasst | Übersieht |
|---|---|---|
| TF-IDF (Skripten 03-05) | exakte Wörter, Namen, Formate | Synonyme, Paraphrasen, unbekannte Wörter |
| statische Wortvektoren | verwandte Wörter | Kontext, seltene Komposita; eingefrorenes Vokabular |
| gemittelte Wortvektoren | grobe Bedeutung des Dokuments | Wortreihenfolge, welche Wörter wichtig sind |
| Sentence Transformer | Bedeutung im Kontext, über Sprachen hinweg | exakte Namen; Schwellenwert muss neu kalibriert werden |

## Übungen

1. **Kombiniere die Signale.** Hänge das Embedding an die TF-IDF-Merkmale aus Skript 05 an (`scipy.sparse.hstack`) und trainiere ein gemeinsames Modell. Schlägt es beide?
2. **Sprachübergreifende Suche.** Formuliere fünf Suchanfragen auf Deutsch und übersetze sie ins Englische und in eine weitere Sprache. Wie viele der sechs besten Treffer stimmen zwischen den Sprachen überein? Bei welchen Anfragen hängt das Ergebnis am stärksten von der Sprache ab?
3. **Kalibrierung des Schwellenwerts.** Bewerte 40 Zuordnungen von Schlagzeilen auf Basis der Embeddings als richtig oder falsch und zeichne die Precision in Abhängigkeit vom Schwellenwert der Ähnlichkeit. Kombiniere dann TF-IDF- und Embedding-Ähnlichkeit (etwa ihren Mittelwert): Steigt die Precision?
4. **Ein größerer Encoder.** Probiere `intfloat/multilingual-e5-base` aus (stelle jedem Text `"query: "` voran, wie es die Modellbeschreibung verlangt). Ist die Suche besser, und was kostet das an Zeit?

## Quellen

* https://spacy.io/models/de
* https://sbert.net/
* https://umap-learn.readthedocs.io/
* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/svm.html
* https://scikit-learn.org/stable/modules/model_evaluation.html